In [20]:
!pip install -q timm==0.4.12 transformers==4.44.2 fairscale==0.4.4 pycocoevalcap ruamel.yaml scipy Pillow
!pip install -q git+https://github.com/openai/CLIP.git

  Preparing metadata (setup.py) ... done


In [30]:

# ===== CONFIG =====
GITHUB_REPO = "https://github.com/ThinhSama234/image_tagging.git"
# ==================

In [31]:
import os
import shutil

os.chdir("/kaggle/working")

# Nếu thư mục tồn tại, xóa sạch nó đi
if os.path.isdir("image_tagging"):
    print("🧹 Removing old cache...")
    shutil.rmtree("image_tagging")

# Clone bản mới nhất
!git clone {GITHUB_REPO}

os.chdir("image_tagging")
!pwd

🧹 Removing old cache...
Cloning into 'image_tagging'...
remote: Enumerating objects: 852, done.
remote: Counting objects: 100% (852/852), done.
remote: Compressing objects: 100% (405/405), done.
remote: Total 852 (delta 460), reused 825 (delta 436), pack-reused 0 (from 0)
Receiving objects: 100% (852/852), 34.84 MiB | 31.49 MiB/s, done.
Resolving deltas: 100% (460/460), done.
/kaggle/working/image_tagging


In [32]:
# ===== KAGGLE INPUT PATHS =====
# Pretrained RAM model (from Kaggle Model: nguyentrthinh/ram-pretrained)
KAGGLE_PRETRAINED = "/kaggle/input/ram-pretrained/pytorch/default/1/ram_swin_large_14m.pth"

# Flickr30k images (from Kaggle Dataset: hsankesara/flickr-image-dataset)
KAGGLE_IMAGES_DIR = "/kaggle/input/flickr-image-dataset/flickr30k_images/flickr30k_images"
# ==============================

In [33]:
import os

# 1. Đường dẫn gốc (FILE XỊN trong Input)
KAGGLE_PRETRAINED_SOURCE = "/kaggle/input/models/nguyentrthinh/ram-pretrained/pytorch/default/1/ram_swin_large_14m.pth"

# 2. Đường dẫn đích (Cái biển chỉ đường trong Repo của bạn)
# Nó phải nằm trong /kaggle/working/image_tagging/pretrained/
local_link_path = "/kaggle/working/image_tagging/pretrained/ram_swin_large_14m.pth"

# 3. Tạo thư mục chứa link nếu chưa có
os.makedirs("/kaggle/working/image_tagging/pretrained", exist_ok=True)

# 4. Xử lý tạo Symlink
if os.path.lexists(local_link_path):
    os.remove(local_link_path) # Xóa link cũ nếu bị lỗi

try:
    os.symlink(KAGGLE_PRETRAINED_SOURCE, local_link_path)
    print(f"✅ Đã tạo liên kết thành công!")
    print(f"🔗 {local_link_path} -> {KAGGLE_PRETRAINED_SOURCE}")
except Exception as e:
    print(f"❌ Lỗi: {e}")

# 5. Kiểm tra lại dung lượng file qua link vừa tạo
!ls -lh /kaggle/working/image_tagging/pretrained/

✅ Đã tạo liên kết thành công!
🔗 /kaggle/working/image_tagging/pretrained/ram_swin_large_14m.pth -> /kaggle/input/models/nguyentrthinh/ram-pretrained/pytorch/default/1/ram_swin_large_14m.pth
total 4.0K
lrwxrwxrwx 1 root root 90 Mar 28 15:34 ram_swin_large_14m.pth -> /kaggle/input/models/nguyentrthinh/ram-pretrained/pytorch/default/1/ram_swin_large_14m.pth


In [25]:
!pwd
!ls -R

/kaggle/working/image_tagging
.:
batch_image		       kaggle_setup.ipynb
batch_inference.py	       LICENSE
datasets		       MANIFEST.in
demo_app		       NOTICE.txt
DOCUMENTATION.md	       pretrained
evaluate.py		       pretrain.py
evaluate_voc.py		       ram
finetune.py		       README.md
generate_tag_des_llm.py        recognize_anything_demo.ipynb
gui_demo.ipynb		       requirements.txt
images			       ROADMAP.md
inference_ram_openset.py       setup.cfg
inference_ram_plus_openset.py  setup.py
inference_ram_plus.py	       TRAINING_GUIDE.md
inference_ram.py	       utils.py
inference_tag2text.py

./batch_image:
base_adapter.py  csv_adapter.py        folder_adapter.py  preprocess.py
config.py	 flickr30k_adapter.py  __init__.py	  voc_adapter.py

./datasets:
flickr30k  hico  imagenet_multi  openimages_common_214	openimages_rare_200

./datasets/flickr30k:
finetune_config_mini.yaml  train.json	    val.json
finetune_config.yaml	   train_mini.json

./datasets/hico:
hico_600_annots.txt  hico_600_t

In [34]:
import os

# Đường dẫn nguồn và đích
KAGGLE_IMAGES_DIR = "/kaggle/input/datasets/hsankesara/flickr-image-dataset/flickr30k_images/flickr30k_images"
dest_link = "/kaggle/working/image_tagging/batch_image/archive/flickr30k_images/flickr30k_images"

# 1. Đảm bảo thư mục cha tồn tại
os.makedirs(os.path.dirname(dest_link), exist_ok=True)

# 2. Xóa link cũ nếu nó đã tồn tại (Dùng lexists để check cả link bị hỏng)
if os.path.lexists(dest_link):
    os.remove(dest_link)
    print(f"🗑️ Đã xóa link cũ để làm mới.")

# 3. Tạo Symlink mới
try:
    os.symlink(KAGGLE_IMAGES_DIR, dest_link)
    print(f"✅ Đã tạo liên kết thành công!")
    print(f"🔗 {dest_link} -> {KAGGLE_IMAGES_DIR}")
except Exception as e:
    print(f"❌ Lỗi: {e}")

# 4. Kiểm tra cuối cùng
!ls {dest_link} | head -n 5

✅ Đã tạo liên kết thành công!
🔗 /kaggle/working/image_tagging/batch_image/archive/flickr30k_images/flickr30k_images -> /kaggle/input/datasets/hsankesara/flickr-image-dataset/flickr30k_images/flickr30k_images
/kaggle/working/image_tagging/batch_image/archive/flickr30k_images/flickr30k_images


In [35]:
# Sửa lỗi yaml.load trong file finetune.py
!sed -i "309s|config = yaml.load(open(args.config, 'r'), Loader=yaml.Loader)|import yaml; config = yaml.safe_load(open(args.config, 'r'))|" /kaggle/working/image_tagging/finetune.py

# Kiểm tra xem đã sửa chưa
!sed -n '309p' /kaggle/working/image_tagging/finetune.py

            torch.save(save_obj, os.path.join(args.output_dir, 'checkpoint_latest.pth'))


In [17]:
!mkdir -p /kaggle/working/pretrained
!cp /kaggle/input/models/nguyentrthinh/ram-pretrained/pytorch/default/1/ram_swin_large_14m.pth /kaggle/working/pretrained/

!python evaluate_voc.py \
    --checkpoint /kaggle/working/pretrained/ram_swin_large_14m.pth \
    --model-type ram \
    --voc-root /kaggle/working/data


/usr/local/lib/python3.12/dist-packages/fairscale/experimental/nn/offload.py:19: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_fwd(orig_func)  # type: ignore
/usr/local/lib/python3.12/dist-packages/fairscale/experimental/nn/offload.py:30: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_bwd(orig_func)  # type: ignore
Device: cuda
Loading VOC 2012 val...
VOC samples: 5823

Loading Model A: /kaggle/working/pretrained/ram_swin_large_14m.pth
/encoder/layer/0/crossattention/self/query is tied
/encoder/layer/0/crossattention/self/key is tied
/encoder/layer/0/crossattention/self/value is tied
/encoder/layer/0/crossattention/output/dense is tied
/encoder/layer/0/crossattention/output/LayerNorm is tied
/encoder/layer/0/intermediate/dense is tied
/encoder

In [36]:
# Preprocess VOC 2012 train → RAM JSON format
!python -m batch_image.preprocess \
    --adapter voc \
    --voc-root /kaggle/working/data \
    --voc-split train \
    --output-dir datasets/pascal_voc_2012 \
    --no-split

<frozen runpy>:128: RuntimeWarning: 'batch_image.preprocess' found in sys.modules after import of package 'batch_image', but prior to execution of 'batch_image.preprocess'; this may result in unpredictable behaviour
Loaded: 5717 raw entries
Valid:  5431 annotations
Skipped (no matching tags): 286
Skipped (image not found):  0
Train: 5431 | Val: 0
Saved: datasets/pascal_voc_2012/train.json (5431 entries)
Config: datasets/pascal_voc_2012/finetune_config.yaml


In [37]:
!python finetune.py \
    --config datasets/pascal_voc_2012/finetune_config.yaml \
    --model-type ram \
    --checkpoint /kaggle/input/models/nguyentrthinh/ram-pretrained/pytorch/default/1/ram_swin_large_14m.pth \
    --output-dir output/voc2012_finetune \
    --distributed False

/usr/local/lib/python3.12/dist-packages/fairscale/experimental/nn/offload.py:19: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_fwd(orig_func)  # type: ignore
/usr/local/lib/python3.12/dist-packages/fairscale/experimental/nn/offload.py:30: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_bwd(orig_func)  # type: ignore
Not using distributed mode
Creating dataset
loading datasets/pascal_voc_2012/train.json
number of training samples: 5431
Creating model
load from: /kaggle/input/models/nguyentrthinh/ram-pretrained/pytorch/default/1/ram_swin_large_14m.pth
Creating pretrained CLIP model
Creating RAM model
/encoder/layer/0/crossattention/self/query is tied
/encoder/layer/0/crossattention/self/key is tied
/encoder/layer/0/crossattention/self/value is ti

In [ ]:
!python evaluate_voc.py \
    --checkpoint /kaggle/input/models/nguyentrthinh/ram-pretrained/pytorch/default/1/ram_swin_large_14m.pth \
    --checkpoint-b output/voc2012_finetune/checkpoint_best.pth \
    --model-type ram \
    --voc-root /kaggle/working/data \
    --output output/voc2012_comparison.json